# 04 Model Training

Training notebook for baseline and advanced models.

In [1]:
# Cell 1: imports
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from evaluation import evaluate_predictions

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

train_linear = pd.read_parquet(processed_dir / "train_linear_ready.parquet")
train_tree = pd.read_parquet(processed_dir / "train_tree_ready.parquet")

print("Linear-ready shape:", train_linear.shape)
print("Tree-ready shape:", train_tree.shape)

Linear-ready shape: (307511, 600)
Tree-ready shape: (307511, 600)


In [3]:
X_linear = train_linear.drop(columns=["TARGET"])
y_linear = train_linear["TARGET"]

X_tree = train_tree.drop(columns=["TARGET"])
y_tree = train_tree["TARGET"]

In [4]:
X_train_linear, X_val_linear, y_train_linear, y_val_linear = train_test_split(
    X_linear, y_linear,
    test_size=0.2,
    random_state=42,
    stratify=y_linear
)

X_train_tree, X_val_tree, y_train_tree, y_val_tree = train_test_split(
    X_tree, y_tree,
    test_size=0.2,
    random_state=42,
    stratify=y_tree
)

print("Linear train shape:", X_train_linear.shape)
print("Linear val shape:", X_val_linear.shape)
print("Tree train shape:", X_train_tree.shape)
print("Tree val shape:", X_val_tree.shape)

Linear train shape: (246008, 599)
Linear val shape: (61503, 599)
Tree train shape: (246008, 599)
Tree val shape: (61503, 599)


In [5]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train_linear, y_train_linear)
log_reg_probs = log_reg.predict_proba(X_val_linear)[:, 1]

log_reg_results = evaluate_predictions(y_val_linear, log_reg_probs)

print("Logistic Regression Results:")
for metric, value in log_reg_results.items():
    print(f"{metric}: {value:.4f}")

Logistic Regression Results:
accuracy: 0.7110
precision: 0.1755
recall: 0.6973
f1: 0.2804
roc_auc: 0.7752
average_precision: 0.2635


In [6]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=20,
    min_samples_leaf=10,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=42
)

rf.fit(X_train_tree, y_train_tree)
rf_probs = rf.predict_proba(X_val_tree)[:, 1]

rf_results = evaluate_predictions(y_val_tree, rf_probs)

print("Random Forest Results:")
for metric, value in rf_results.items():
    print(f"{metric}: {value:.4f}")

Random Forest Results:
accuracy: 0.8931
precision: 0.3069
recall: 0.2578
f1: 0.2802
roc_auc: 0.7664
average_precision: 0.2447


In [7]:
results_df = pd.DataFrame([
    {"model": "Logistic Regression", **log_reg_results},
    {"model": "Random Forest", **rf_results},
])

results_df

,model,accuracy,precision,recall,f1,roc_auc,average_precision
0,Logistic Regression,0.711022,0.175451,0.697281,0.280358,0.775222,0.263518
1,Random Forest,0.893078,0.306881,0.257805,0.280210,0.766405,0.244679
